# [Vectorizers in BERTopic](https://maartengr.github.io/BERTopic/getting_started/vectorizers/vectorizers.html)


In BERTopic, **vectorizers** are crucial for transforming textual data into numerical representations, enabling the extraction of meaningful topics. The quality of these vectorizations directly impacts the interpretability and coherence of the derived topics. BERTopic offers flexibility in choosing and customizing vectorization techniques to suit various project requirements. 

### **CountVectorizer**

The `CountVectorizer` converts a collection of text documents into a matrix of token counts, serving as the foundation for the c-TF-IDF calculation in BERTopic. It offers several parameters for customization:

- **`ngram_range`**: Defines the range of n-values for n-grams to be extracted, allowing the capture of more context within the text.

- **`stop_words`**: Specifies words to be removed from the text before processing, helping to eliminate common but uninformative words.

- **`min_df`**: Sets the minimum frequency threshold for words to be included in the vocabulary, filtering out rare terms.

- **`max_features`**: Limits the number of features (tokens) to the most frequent ones, controlling the dimensionality of the feature space.

Customizing these parameters allows for fine-tuning topic representations without retraining the entire model.

### **OnlineCountVectorizer**

The `OnlineCountVectorizer` is an online variant that updates its vocabulary incrementally, suitable for streaming data or scenarios requiring online learning. It includes additional parameters:

- **`decay`**: A value between [0, 1] that weights the percentage by which the frequencies in the bag-of-words matrix should decrease at each iteration, allowing the model to adapt to new data over time.

- **`delete_min_df`**: Removes words from the vocabulary that fall below a minimum frequency threshold during updates, preventing the vocabulary from growing indefinitely.

This flexibility allows for fine-tuning topic representations without retraining the entire model. 

### **Choosing the Appropriate Vectorization Technique**

- **CountVectorizer**: Ideal for static datasets where the entire corpus is available upfront. It allows for extensive customization to tailor the vectorization process to the specific characteristics of the dataset.

- **OnlineCountVectorizer**: Suited for dynamic or streaming data environments where new documents arrive continuously. Its ability to update the vocabulary incrementally makes it valuable for applications requiring real-time topic modeling.

Selecting the appropriate vectorizer and tuning its parameters are essential steps in optimizing the performance and interpretability of topic models in BERTopic.

## CountVectorizer

One often underestimated component of BERTopic is the CountVectorizer and c-TF-IDF calculation. Together, they are responsible for creating the topic representations and luckily can be quite flexible in parameter tuning. Here, we will go through tips and tricks for tuning your CountVectorizer and see how they might affect the topic representations.

Before starting, it should be noted that you can pass the CountVectorizer before and after training your topic model. Passing it before training allows you to minimize the size of the resulting c-TF-IDF matrix:

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Train BERTopic with a custom CountVectorizer
vectorizer_model = CountVectorizer(min_df=10)
topic_model = BERTopic(vectorizer_model=vectorizer_model)
topics, probs = topic_model.fit_transform(docs)

Passing it after training allows you to fine-tune the topic representations by using .update_topics():

In [ ]:
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# Train a BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

# Fine-tune topic representations after training BERTopic
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 3), min_df=10)
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

The great thing about using .update_topics() is that it allows you to tweak the topic representations without re-training your model! Thus, here we will be focusing on fine-tuning our topic representations after training our model.

# CountVectorizer

First, let's start with defining our documents and training our topic model:

In [2]:
from bertopic import BERTopic
from sklearn.datasets import fetch_20newsgroups

# Prepare documents
docs = fetch_20newsgroups(subset='all',  remove=('headers', 'footers', 'quotes'))['data']

# Train a BERTopic model
topic_model = BERTopic()
topics, probs = topic_model.fit_transform(docs)

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitl

Now, let's see the top 10 most frequent topics that have been generated:

In [3]:
topic_model.get_topic_info()[1:11]

,Topic,Count,Name,Representation,Representative_Docs
1,0,690,0_he_game_year_hit,"[he, game, year, hit, baseball, players, team,...",[\nNot particularly *in* the World Series. Dur...
2,1,564,1_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, escrow,...",[Here is a revised version of my summary which...
3,2,528,2_idjits_ites_cheek_dancing,"[idjits, ites, cheek, dancing, yep, huh, ken, ...","[Ken\n, \n \n ..."
4,3,460,3_israel_israeli_jews_arab,"[israel, israeli, jews, arab, arabs, jewish, p...",[From: Center for Policy Research <cpr>\nSubje...
5,4,440,4_drive_scsi_drives_ide,"[drive, scsi, drives, ide, disk, controller, h...",[Thanks to all who responded to my original po...
6,5,431,5_monitor_card_video_drivers,"[monitor, card, video, drivers, vga, monitors,...",[Hi there. We just bought a 486 DX2/66 Gatewa...
7,6,415,6_you_your_post_that,"[you, your, post, that, context, jim, me, not,...","[New in this version: challenge #5, plus an a..."
8,7,305,7_car_cars_engine_ford,"[car, cars, engine, ford, mustang, v8, toyota,...","[\n\n\n\nYou know, I'm a Ford fan, I must say,..."
9,8,263,8_health_list_newsgroup_cancer,"[health, list, newsgroup, cancer, disease, tob...",[------------- cut here -----------------\nVol...
10,9,220,9_ram_sale_drive_price,"[ram, sale, drive, price, os, monitor, system,...","[Hello, I have a motherboard and a case for sa..."


The topic representations generated already seem quite interpretable! However, I am quite sure we do much better without having to re-train our model. Next, we will go through common parameters in CountVectorizer and focus on the effects that they might have. As a baseline, we will be comparing them to the topic representation above.

## Parameters
There are several basic parameters in the CountVectorizer that we can use to improve upon the quality of the resulting topic representations.

### **ngram_range**

The ngram_range parameter allows us to decide how many tokens each entity is in a topic representation. For example, we have words like game and team with a length of 1 in a topic but it would also make sense to have words like hockey league with a length of 2. To allow for these words to be generated, we can set the ngram_range parameter:

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(ngram_range=(1, 3), stop_words="english")
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

As you might have noticed, I also added stop_words="english". This is necessary as longer words tend to have many stop words and removing them allows for nicer topic representations:

In [8]:
topic_model.get_topic_info()[1:11]

,Topic,Count,Name,Representation,Representative_Docs
1,0,690,0_year_game_hit_baseball,"[year, game, hit, baseball, team, players, gam...",[\nNot particularly *in* the World Series. Dur...
2,1,564,1_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, escrow,...",[Here is a revised version of my summary which...
3,2,528,2_good cheek_dancing idjits_ites 15_idjits goo...,"[good cheek, dancing idjits, ites 15, idjits g...","[Ken\n, \n \n ..."
4,3,460,3_israel_israeli_jews_arab,"[israel, israeli, jews, arab, jewish, arabs, p...",[From: Center for Policy Research <cpr>\nSubje...
5,4,440,4_drive_scsi_drives_ide,"[drive, scsi, drives, ide, disk, controller, h...",[Thanks to all who responded to my original po...
6,5,431,5_card_monitor_video_drivers,"[card, monitor, video, drivers, vga, monitors,...",[Hi there. We just bought a 486 DX2/66 Gatewa...
7,6,415,6_post_context_jim_deleted,"[post, context, jim, deleted, say, dont, im, f...","[New in this version: challenge #5, plus an a..."
8,7,305,7_car_cars_engine_ford,"[car, cars, engine, ford, mustang, toyota, v8,...","[\n\n\n\nYou know, I'm a Ford fan, I must say,..."
9,8,263,8_health_list_newsgroup_cancer,"[health, list, newsgroup, cancer, email, disea...",[------------- cut here -----------------\nVol...
10,9,220,9_ram_drive_sale_price,"[ram, drive, sale, price, monitor, os, softwar...","[Hello, I have a motherboard and a case for sa..."


Although they look very similar, if we zoom in on topic 8, we can see longer words in our representation:

In [15]:
topic_model.get_topic(8)

[('health', 0.005841362973043772),
 ('list', 0.005808500484592155),
 ('newsgroup', 0.005062636421989067),
 ('cancer', 0.0047521520149178985),
 ('email', 0.00418069969151725),
 ('disease', 0.004094418009565857),
 ('send', 0.004039864057008132),
 ('1993', 0.004007689564193538),
 ('medical', 0.003974601020157142),
 ('tobacco', 0.00394823346621094)]

### **stop_words**

In some of the topics, we can see stop words appearing like he or the.
Stop words are something we typically want to prevent in our topic representations as they do not give additional information to the topic. To prevent those stop words, we can use the stop_words parameter in the CountVectorizer to remove them from the representations:

In [17]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(stop_words="english")
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)

After running the above, we get the following output:

In [18]:
topic_model.get_topic_info()[1:11]

,Topic,Count,Name,Representation,Representative_Docs
1,0,690,0_game_year_baseball_hit,"[game, year, baseball, hit, players, team, pit...",[\nNot particularly *in* the World Series. Dur...
2,1,564,1_key_clipper_encryption_chip,"[key, clipper, encryption, chip, keys, escrow,...",[Here is a revised version of my summary which...
3,2,528,2_idjits_ites_cheek_dancing,"[idjits, ites, cheek, dancing, yep, huh, ken, ...","[Ken\n, \n \n ..."
4,3,460,3_israel_israeli_jews_arab,"[israel, israeli, jews, arab, arabs, jewish, p...",[From: Center for Policy Research <cpr>\nSubje...
5,4,440,4_drive_scsi_drives_ide,"[drive, scsi, drives, ide, controller, disk, s...",[Thanks to all who responded to my original po...
6,5,431,5_monitor_card_video_drivers,"[monitor, card, video, drivers, vga, monitors,...",[Hi there. We just bought a 486 DX2/66 Gatewa...
7,6,415,6_post_context_jim_deleted,"[post, context, jim, deleted, ted, frank, quot...","[New in this version: challenge #5, plus an a..."
8,7,305,7_car_cars_engine_ford,"[car, cars, engine, ford, mustang, v8, toyota,...","[\n\n\n\nYou know, I'm a Ford fan, I must say,..."
9,8,263,8_health_list_newsgroup_cancer,"[health, list, newsgroup, cancer, tobacco, dis...",[------------- cut here -----------------\nVol...
10,9,220,9_ram_sale_os_drive,"[ram, sale, os, drive, price, monitor, meg, mo...","[Hello, I have a motherboard and a case for sa..."


As you can see, the topic representations already look much better! Stop words are removed and the representations are more interpretable. We can also pass in a list of stop words if you have multiple languages to take into account.

### **min_df**

One important parameter to keep in mind is the min_df. This is typically an integer representing how frequent a word must be before being added to our representation. You can imagine that if we have a million documents and a certain word only appears a single time across all of them, then it would be highly unlikely to be representative of a topic. Typically, the c-TF-IDF calculation removes that word from the topic representation but when you have millions of documents, that will also lead to a very large topic-term matrix. To prevent a huge vocabulary, we can set the min_df to only accept words that have a minimum frequency.

When you have millions of documents or error issues, I would advise increasing the value of min_df as long as the topic representations might sense:

In [19]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(min_df=10)
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)


With the following topic representation:

In [20]:
topic_model.get_topic_info()[1:11]

,Topic,Count,Name,Representation,Representative_Docs
1,0,690,0_game_year_he_hit,"[game, year, he, hit, baseball, players, team,...",[\nNot particularly *in* the World Series. Dur...
2,1,564,1_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, governm...",[Here is a revised version of my summary which...
3,2,528,2_dancing_yep_huh_ken,"[dancing, yep, huh, ken, art, why, 15, each, v...","[Ken\n, \n \n ..."
4,3,460,3_israel_jews_arab_jewish,"[israel, jews, arab, jewish, peace, not, of, e...",[From: Center for Policy Research <cpr>\nSubje...
5,4,440,4_drive_scsi_drives_ide,"[drive, scsi, drives, ide, disk, controller, h...",[Thanks to all who responded to my original po...
6,5,431,5_monitor_card_video_drivers,"[monitor, card, video, drivers, vga, monitors,...",[Hi there. We just bought a 486 DX2/66 Gatewa...
7,6,415,6_you_your_post_context,"[you, your, post, context, jim, that, me, dele...","[New in this version: challenge #5, plus an a..."
8,7,305,7_car_cars_engine_ford,"[car, cars, engine, ford, toyota, miles, power...","[\n\n\n\nYou know, I'm a Ford fan, I must say,..."
9,8,263,8_health_list_newsgroup_cancer,"[health, list, newsgroup, cancer, tobacco, dis...",[------------- cut here -----------------\nVol...
10,9,220,9_ram_sale_drive_os,"[ram, sale, drive, os, price, monitor, meg, pc...","[Hello, I have a motherboard and a case for sa..."


As you can see, the output is nearly the same which is what we would like to achieve. All words that appear less than 10 times are now removed from our topic-term matrix (i.e., c-TF-IDF matrix) which drastically lowers the matrix in size.

### **max_features**

A parameter similar to min_df is max_features which allows you to select the top n most frequent words to be used in the topic representation. Setting this, for example, to 10_000 creates a topic-term matrix with 10_000 terms. This helps you control the size of the topic-term matrix directly without having to fiddle around with the min_df parameter:

In [21]:
from sklearn.feature_extraction.text import CountVectorizer
vectorizer_model = CountVectorizer(max_features=10_000)
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)


With the following representation:

In [22]:
topic_model.get_topic_info()[1:11]

,Topic,Count,Name,Representation,Representative_Docs
1,0,690,0_game_he_year_hit,"[game, he, year, hit, baseball, players, team,...",[\nNot particularly *in* the World Series. Dur...
2,1,564,1_key_clipper_chip_encryption,"[key, clipper, chip, encryption, keys, escrow,...",[Here is a revised version of my summary which...
3,2,528,2_yep_huh_ken_art,"[yep, huh, ken, art, why, 15, each, very, good...","[Ken\n, \n \n ..."
4,3,460,3_israel_israeli_jews_arab,"[israel, israeli, jews, arab, arabs, jewish, p...",[From: Center for Policy Research <cpr>\nSubje...
5,4,440,4_drive_scsi_drives_ide,"[drive, scsi, drives, ide, disk, controller, h...",[Thanks to all who responded to my original po...
6,5,431,5_monitor_card_video_drivers,"[monitor, card, video, drivers, vga, monitors,...",[Hi there. We just bought a 486 DX2/66 Gatewa...
7,6,415,6_you_your_post_context,"[you, your, post, context, jim, that, me, my, ...","[New in this version: challenge #5, plus an a..."
8,7,305,7_car_cars_engine_ford,"[car, cars, engine, ford, mustang, v8, toyota,...","[\n\n\n\nYou know, I'm a Ford fan, I must say,..."
9,8,263,8_health_list_newsgroup_cancer,"[health, list, newsgroup, cancer, disease, tob...",[------------- cut here -----------------\nVol...
10,9,220,9_ram_sale_drive_price,"[ram, sale, drive, price, os, monitor, meg, sy...","[Hello, I have a motherboard and a case for sa..."


As with min_df, we would like the topic representations to be very similar.

### **tokenizer**

The default tokenizer in the CountVectorizer works well for western languages but fails to tokenize some non-western languages, like Chinese. Fortunately, we can use the tokenizer variable in the CountVectorizer to use jieba, which is a package for Chinese text segmentation. Using it is straightforward:



In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import jieba

def tokenize_zh(text):
    words = jieba.lcut(text)
    return words

vectorizer = CountVectorizer(tokenizer=tokenize_zh)

Then, we can simply pass the vectorizer to update our topic representations:

In [ ]:
topic_model.update_topics(docs, vectorizer_model=vectorizer_model)


# OnlineCountVectorizer

When using the online/incremental variant of BERTopic, we need a CountVectorizer than can incrementally update its representation. For that purpose, OnlineCountVectorizer was created that not only updates out-of-vocabulary words but also implements decay and cleaning functions to prevent the sparse bag-of-words matrix to become too large. It is a class that can be found in bertopic.vectorizers which extends sklearn.feature_extraction.text.CountVectorizer. In other words, you can use the exact same parameter in OnlineCountVectorizer as found in Scikit-Learn's CountVectorizer. We can use it as follows:

In [23]:
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer

# Train BERTopic with a custom OnlineCountVectorizer
vectorizer_model = OnlineCountVectorizer()
topic_model = BERTopic(vectorizer_model=vectorizer_model)


## Parameters
Other than parameters found in CountVectorizer, such as stop_words and ngram_range, we have two parameters in OnlineCountVectorizer to adjust the way old data is processed and kept.

### **decay**
At each iteration, we sum the bag-of-words representation of the new documents with the bag-of-words representation of all documents processed thus far. In other words, the bag-of-words matrix keeps increasing with each iteration. However, especially in a streaming setting, older documents might become less and less relevant as time goes on. Therefore, a decay parameter was implemented that decays the bag-of-words' frequencies at each iteration before adding the document frequencies of new documents. The decay parameter is a value between 0 and 1 and indicates the percentage of frequencies the previous bag-of-words matrix should be reduced to. For example, a value of .1 will decrease the frequencies in the bag-of-words matrix by 10% at each iteration before adding the new bag-of-words matrix. This will make sure that recent data has more weight than previous iterations.

### **delete_min_df**
In BERTopic, we might want to remove words from the topic representation that appear infrequently. The min_df in the CountVectorizer works quite well for that. However, when we have a streaming setting, the min_df does not work as well since a word's frequency might start below min_df but will end up higher than that over time. Setting that value high might not always be advised.

As a result, the vocabulary of the resulting bag-of-words matrix can become quite large. Similarly, if we implement the decay parameter, then some values will decrease over time until they are below min_df. For these reasons, the delete_min_df parameter was implemented. The parameter takes positive integers and indicates, at each iteration, which words will be removed. If the value is set to 5, it will check after each iteration if the total frequency of a word is exceeded by that value. If so, the word will be removed in its entirety from the bag-of-words matrix. This helps to keep the bag-of-words matrix of a manageable size.